<!-- mccl/notebooks/continual_benchmark.ipynb -->
# Benchmark d'apprentissage continu — 28 classes

**Objectif :** mesurer l'efficacité des 3 méthodes les plus robustes (Replay, EWC, DualMemory) sur un dataset à 28 classes.

**Protocole :**
1. 28 classes au total, ordre aléatoire (seed fixe)
2. Tâche 0 = 2 classes, puis +2 classes par tâche → **14 tâches**
3. Trois stratégies comparées sur les mêmes tâches
4. **Détection du point de rupture** : à partir de quelle tâche l'oubli devient inacceptable
5. **Analyse des faiblesses** : quelles classes sont oubliées, quelles stratégies échouent

**Critère d'oubli catastrophique :** drop > 20 % sur une classe ancienne entre son pic et sa valeur finale.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import copy
import pickle
import zipfile
import json
import time

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    import src.outils.data_loader  # noqa: F401
except ImportError as e:
    print("⚠️ outils.data_loader :", e)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from tqdm import tqdm

from src.config import ModelCfg, ManifoldCfg, RegCfg
from src.models.jepa import JepaModel
from src.training.optim import build_optimizer
from src.training.losses import classification_loss
from src.training.trainer import train_one_task
from src.evaluation.metrics import per_class_accuracy
from src.continual.strategies import build_strategy
from src.utils.seed import set_seed
from src.utils.logging import get_logger

set_seed(0)
log = get_logger("bench")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("ROOT   :", ROOT)
print("device :", device)
print("torch  :", torch.__version__)

## 2. Chargement du dataset 28 classes

Adapter `ZIP_PATH` à votre fichier. Le dataset doit contenir 28 classes.

In [ ]:
def load_zip(zip_path, internal="loader.pkl"):
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(internal) as f:
            l = pickle.load(f, encoding="latin1")
    if l.data_ is None:
        l.data_ = copy.deepcopy(l.original_data_)
    return l


# Adapter le chemin
ZIP_PATH = ROOT / "data" / "dataset28.zip"
if not ZIP_PATH.exists():
    ZIP_PATH = ROOT / "dataset28.zip"

loader = load_zip(ZIP_PATH)
real_df = loader.data_

if not getattr(loader, "class_mapping_", None):
    loader.class_mapping_ = {c: i for i, c in enumerate(loader.name_label_)}

n_classes = len(loader.name_label_)
print(f"Shape      : {real_df.shape}")
print(f"Classes    : {n_classes}")
print(f"Liste      : {loader.name_label_[:10]}...")
print(f"Mapping OK : {len(loader.class_mapping_) == n_classes}")

assert n_classes == 28, f"Attendu 28 classes, trouvé {n_classes}"

## 3. Split 70/15/15

In [ ]:
from sklearn.model_selection import train_test_split

labels = real_df["Label"].map(loader.class_mapping_).to_numpy(dtype=int)
idx = np.arange(len(real_df))

tr, tmp = train_test_split(idx, test_size=0.30, stratify=labels, random_state=0)
va, te = train_test_split(tmp, test_size=0.50, stratify=labels[tmp], random_state=0)

print(f"train={len(tr)} | val={len(va)} | test={len(te)}")
print(f"Classes par split (train) : min={np.bincount(labels[tr]).min()}, max={np.bincount(labels[tr]).max()}")

## 4. Dataset + transforms

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

transform_train = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

transform_eval = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class ImageSubset(Dataset):
    def __init__(self, images, labels, indices, transform=None):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        img = self.images[j]

        if not isinstance(img, Image.Image):
            arr = np.asarray(img)
            if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[0] != arr.shape[-1]:
                arr = np.transpose(arr, (1, 2, 0))
            if arr.dtype != np.uint8:
                arr = (arr * 255).clip(0, 255).astype(np.uint8) if arr.max() <= 1.0 \
                      else arr.clip(0, 255).astype(np.uint8)
            if arr.ndim == 3 and arr.shape[2] == 1:
                arr = arr[:, :, 0]
            img = Image.fromarray(arr)

        if img.mode != "RGB":
            img = img.convert("RGB")

        if self.transform is not None:
            img = self.transform(img)
        else:
            img = T.ToTensor()(img)

        return img, int(self.labels[j])


images = real_df["Image"].tolist()

## 5. Ordre des tâches

Tâche 0 = 2 classes, puis +2 classes par tâche.

Avec 28 classes : 14 tâches au total.

- Tâche 0 : classes {a, b}
- Tâche 1 : classes {a, b, c, d}
- ...
- Tâche 13 : toutes les 28 classes

In [ ]:
CLASSES_PER_TASK = 2
FIRST_TASK_SIZE = 2

# Ordre aléatoire reproductible
rng = np.random.default_rng(42)
class_order = rng.permutation(n_classes).tolist()
print("Ordre des classes :", class_order)

# Construire les tâches
tasks = []
consumed = 0
task_idx = 0
while consumed < n_classes:
    if task_idx == 0:
        size = FIRST_TASK_SIZE
    else:
        size = CLASSES_PER_TASK
    new_classes = class_order[consumed:consumed + size]
    if not new_classes:
        break
    consumed += len(new_classes)
    tasks.append({
        "idx": task_idx,
        "classes": new_classes,
        "seen": class_order[:consumed],
    })
    task_idx += 1

print(f"\n{len(tasks)} tâches :")
for t in tasks:
    print(f"  Tâche {t['idx']:2d} : new={t['classes']}, total_vues={len(t['seen'])}")

## 6. Fonctions d'entraînement et d'évaluation

In [ ]:
def make_cfg_euclidean():
    return ModelCfg(
        name="euclid",
        manifold=ManifoldCfg(
            type="product",
            components=[{"type": "euclidean", "dim": 192}],
            pooling="mean",
        ),
        reg=RegCfg(lambda_h=0.0, lambda_s=0.0, lambda_e=0.0),
    )


@torch.no_grad()
def eval_per_class(model, loader, n_cls):
    model.eval()
    all_logits, all_y = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        out = model(xb)
        if out.logits is not None:
            all_logits.append(out.logits.cpu())
            all_y.append(yb)
    if not all_logits:
        return {}
    logits = torch.cat(all_logits, 0)
    y = torch.cat(all_y, 0)
    pc = per_class_accuracy(logits, y, n_cls)
    return {c: float(pc[c]) for c in range(n_cls) if not np.isnan(pc[c])}


def run_strategy(strategy_name, n_epochs=3, batch_size=32, lr=1e-3, seed=0):
    """Lance une stratégie sur les 14 tâches."""
    set_seed(seed)
    cfg = make_cfg_euclidean()
    model = JepaModel(cfg).to(device)
    opt = build_optimizer(model, lr=lr)

    # Stratégie
    if strategy_name == "replay":
        strategy = build_strategy("replay", capacity=2000)
    elif strategy_name == "ewc":
        strategy = build_strategy("ewc", lam=1e4, n_samples=256)
        strategy.attach_model(model)
    elif strategy_name == "dual_memory":
        strategy = build_strategy("dual_memory",
                                  fast_cfg={"capacity": 1000},
                                  slow_cfg={"capacity_per_class": 64})
    else:
        raise ValueError(strategy_name)

    # Matrice d'accuracy [T, n_classes] (NaN pour non vu)
    T = len(tasks)
    acc_matrix = np.full((T, n_classes), np.nan)

    t_start = time.time()
    for task in tasks:
        t_idx = task["idx"]
        new_classes = task["classes"]
        model.register_new_classes(new_classes)
        strategy.begin_task(type("T", (), {"index": t_idx, "classes": new_classes})())

        # Loader d'entraînement (uniquement les nouvelles classes)
        train_mask = np.isin(labels[tr], new_classes)
        train_idx = tr[train_mask]
        ds_t = ImageSubset(images, labels, train_idx, transform_train)
        dl_t = DataLoader(ds_t, batch_size=batch_size, shuffle=True, num_workers=0)

        def loss_fn(m, batch, out):
            xb, yb = batch
            loss = classification_loss(out.logits, yb.to(device))
            replay = strategy.sample_replay(xb.size(0))
            if replay is not None:
                xr, yr = replay
                out_r = m(xr.to(device))
                loss = loss + classification_loss(out_r.logits, yr.to(device))
            pen = strategy.penalty()
            if pen.requires_grad or pen.item() > 0:
                loss = loss + pen
            return loss

        def on_step(state, batch):
            xb, yb = batch
            with torch.no_grad():
                out = model(xb.to(device))
            strategy.observe((xb.detach().cpu(), yb.detach().cpu()), out=out)

        train_one_task(
            model=model, train_loader=dl_t, optimizer=opt,
            n_epochs=n_epochs, loss_fn=loss_fn, device=device,
            on_step=on_step, task_idx=t_idx, verbose=False,
        )

        strategy.end_task(type("T", (), {"index": t_idx, "classes": new_classes})(),
                          train_loader=dl_t)

        # Évaluation sur les classes vues
        test_mask = np.isin(labels[te], task["seen"])
        test_idx = te[test_mask]
        ds_test = ImageSubset(images, labels, test_idx, transform_eval)
        dl_test = DataLoader(ds_test, batch_size=64, shuffle=False, num_workers=0)
        per_class = eval_per_class(model, dl_test, n_classes)

        for c, acc in per_class.items():
            acc_matrix[t_idx, c] = acc

        overall = np.nanmean([acc_matrix[t_idx, c] for c in task["seen"]])
        print(f"  Tâche {t_idx:2d} | new={new_classes} | "
              f"overall={overall:.4f} | {time.time()-t_start:.1f}s")

    return model, strategy, acc_matrix, time.time() - t_start

## 7. Lancer les 3 stratégies

⚠️ **Durée estimée** :
- Sur CPU : ~2-4 h par stratégie
- Sur GPU : ~20-40 min par stratégie

Vous pouvez lancer une stratégie à la fois pour éviter les timeouts.

In [ ]:
STRATEGIES = ["replay", "ewc", "dual_memory"]
results = {}

for strat in STRATEGIES:
    print("\n" + "=" * 70)
    print(f"STRATÉGIE : {strat}")
    print("=" * 70)
    model, strategy, acc_matrix, duration = run_strategy(
        strat, n_epochs=3, batch_size=32, seed=0,
    )
    results[strat] = {
        "acc_matrix": acc_matrix,
        "duration": duration,
        "strategy_state": strategy.state() if hasattr(strategy, "state") else {},
    }
    print(f"✓ {strat} terminé en {duration:.1f}s")

## 8. Analyse du point de rupture

**Définition de l'oubli catastrophique :**

Pour chaque classe c, on regarde :
- `peak[c]` = accuracy maximale de c juste après son apprentissage
- `final[c]` = accuracy de c après la dernière tâche
- `drop[c] = peak[c] - final[c]`

Si `drop[c] > 0.20` (20 points), c'est un oubli catastrophique.

Le **point de rupture** est la première tâche où plus de 30 % des classes anciennes ont subi un oubli catastrophique.

In [ ]:
def analyze_breakpoint(acc_matrix, threshold=0.20):
    """Détecte la tâche où l'oubli devient inacceptable."""
    T, C = acc_matrix.shape
    breakpoint = None
    per_task_stats = []

    for t in range(1, T):
        drops = []
        for c in range(C):
            col = acc_matrix[:t + 1, c]
            valid = ~np.isnan(col)
            if valid.sum() < 2:
                continue
            peak = np.nanmax(col)
            current = col[-1]
            drops.append(peak - current)

        if not drops:
            per_task_stats.append({
                "task": t, "mean_drop": 0.0,
                "frac_forgotten": 0.0,
            })
            continue

        drops = np.array(drops)
        mean_drop = float(drops.mean())
        frac_forgotten = float((drops > threshold).mean())

        per_task_stats.append({
            "task": t,
            "mean_drop": mean_drop,
            "frac_forgotten": frac_forgotten,
        })

        if breakpoint is None and frac_forgotten > 0.30:
            breakpoint = t

    return breakpoint, per_task_stats


print("=" * 70)
print("ANALYSE DU POINT DE RUPTURE")
print("=" * 70)

for strat in STRATEGIES:
    mat = results[strat]["acc_matrix"]
    bp, stats = analyze_breakpoint(mat, threshold=0.20)
    results[strat]["breakpoint"] = bp
    results[strat]["per_task_stats"] = stats

    if bp is None:
        print(f"{strat:15s} : jamais de rupture (oubli < 20pts)")
    else:
        print(f"{strat:15s} : rupture à la tâche {bp}")

    # Moyenne finale
    final_acc = np.nanmean(mat[-1])
    print(f"  Accuracy finale moyenne : {final_acc:.4f}")

## 9. Tableau récapitulatif

In [ ]:
rows = []
for strat in STRATEGIES:
    mat = results[strat]["acc_matrix"]
    bp = results[strat]["breakpoint"]

    # Accuracy moyenne par tâche
    per_task_acc = []
    for t in range(mat.shape[0]):
        row = mat[t]
        valid = ~np.isnan(row)
        if valid.sum() > 0:
            per_task_acc.append(np.nanmean(row[valid]))
        else:
            per_task_acc.append(np.nan)

    # Oubli moyen final
    T = mat.shape[0]
    drops = []
    for c in range(mat.shape[1]):
        col = mat[:, c]
        valid = ~np.isnan(col)
        if valid.sum() >= 2:
            peak = np.nanmax(col)
            final = col[-1]
            drops.append(peak - final)
    mean_forget = float(np.mean(drops)) if drops else 0.0

    rows.append({
        "Stratégie":           strat,
        "Breakpoint":          bp if bp is not None else "—",
        "Avg acc finale":      round(float(np.nanmean(mat[-1])), 4),
        "Avg acc peak":        round(float(np.nanmax(per_task_acc)), 4),
        "Oubli moyen":         round(mean_forget, 4),
        "Durée (s)":           round(results[strat]["duration"], 1),
    })

df = pd.DataFrame(rows)
print("\n" + "=" * 90)
print("TABLEAU RÉCAPITULATIF")
print("=" * 90)
print(df.to_string(index=False))
print("=" * 90)

out_dir = ROOT / "results"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "continual_benchmark.csv", index=False)
print(f"\nSauvegardé : {out_dir / 'continual_benchmark.csv'}")

## 10. Figure 1 — Accuracy moyenne par tâche

In [ ]:
colors = {
    "replay":      "#0072B2",
    "ewc":         "#D55E00",
    "dual_memory": "#CC79A7",
}

fig, ax = plt.subplots(figsize=(10, 5))

for strat in STRATEGIES:
    mat = results[strat]["acc_matrix"]
    per_task = []
    for t in range(mat.shape[0]):
        valid = ~np.isnan(mat[t])
        per_task.append(np.nanmean(mat[t][valid]) if valid.sum() > 0 else np.nan)

    ax.plot(range(len(per_task)), per_task,
            marker="o", markersize=6, linewidth=2.2,
            color=colors[strat], label=strat,
            markerfacecolor="white", markeredgewidth=2)

    # Marquer le breakpoint
    bp = results[strat]["breakpoint"]
    if bp is not None:
        ax.axvline(bp, color=colors[strat], linestyle="--", alpha=0.4)

ax.set_xlabel("Tâche")
ax.set_ylabel("Accuracy moyenne sur classes vues")
ax.set_title("Évolution de l'accuracy au fil des tâches", loc="left")
ax.legend(loc="best", frameon=False)
ax.grid(alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(out_dir / "benchmark_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Figure 2 — Oubli par classe

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)

for ax, strat in zip(axes, STRATEGIES):
    mat = results[strat]["acc_matrix"]
    C = mat.shape[1]
    drops = []
    for c in range(C):
        col = mat[:, c]
        valid = ~np.isnan(col)
        if valid.sum() >= 2:
            peak = np.nanmax(col)
            final = col[-1]
            drops.append(peak - final)
        else:
            drops.append(0.0)

    class_names = loader.name_label_
    x = np.arange(len(drops))
    colors_bar = ["#D55E00" if d > 0.20 else "#0072B2" for d in drops]

    ax.bar(x, drops, color=colors_bar, edgecolor="white", linewidth=0.8)
    ax.axhline(0.20, color="#D55E00", linestyle="--", linewidth=1,
               label="Seuil 20%")
    ax.set_title(strat, loc="left")
    ax.set_xlabel("Classe")
    ax.set_xticks(x[::2])
    ax.set_xticklabels([class_names[i][:6] for i in x[::2]],
                       rotation=45, ha="right", fontsize=7)
    if ax is axes[0]:
        ax.set_ylabel("Oubli (peak - final)")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Oubli par classe — les classes en rouge dépassent 20%",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(out_dir / "benchmark_forgetting.png", dpi=300, bbox_inches="tight")
plt.show()

## 12. Figure 3 — Fraction de classes oubliées par tâche

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for strat in STRATEGIES:
    stats = results[strat]["per_task_stats"]
    tasks = [s["task"] for s in stats]
    fracs = [s["frac_forgotten"] for s in stats]
    ax.plot(tasks, fracs, marker="o", markersize=6, linewidth=2.2,
            color=colors[strat], label=strat,
            markerfacecolor="white", markeredgewidth=2)

ax.axhline(0.30, color="red", linestyle="--", linewidth=1,
           label="Seuil de rupture (30%)")
ax.set_xlabel("Tâche")
ax.set_ylabel("Fraction de classes avec oubli > 20%")
ax.set_title("Détection du point de rupture", loc="left")
ax.legend(loc="best", frameon=False)
ax.grid(alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(out_dir / "benchmark_breakpoint.png", dpi=300, bbox_inches="tight")
plt.show()

## 13. Analyse des faiblesses

In [ ]:
print("=" * 75)
print("ANALYSE DES FAIBLESSES")
print("=" * 75)

for strat in STRATEGIES:
    print(f"\n{strat.upper()} :")
    mat = results[strat]["acc_matrix"]
    bp = results[strat]["breakpoint"]

    if bp is None:
        print("  ✓ Pas de rupture détectée")
    else:
        print(f"  ✗ Rupture à la tâche {bp} ({bp/len(tasks)*100:.0f}% du stream)")

    # Classes les plus oubliées
    drops = []
    for c in range(mat.shape[1]):
        col = mat[:, c]
        valid = ~np.isnan(col)
        if valid.sum() >= 2:
            peak = np.nanmax(col)
            final = col[-1]
            drops.append((c, peak - final, peak, final))

    drops.sort(key=lambda x: x[1], reverse=True)

    print("  Top 5 classes les plus oubliées :")
    for c, drop, peak, final in drops[:5]:
        name = loader.name_label_[c]
        print(f"    {name:15s} : peak={peak:.2f} → final={final:.2f} (drop={drop:.2f})")

    # Classes bien retenues
    print("  Top 3 classes les mieux retenues :")
    for c, drop, peak, final in drops[-3:]:
        name = loader.name_label_[c]
        print(f"    {name:15s} : peak={peak:.2f} → final={final:.2f} (drop={drop:.2f})")

print("\n" + "=" * 75)

## 14. Conclusion et pistes pour une nouvelle méthode

**Ce qu'on cherche :**
1. Quelle stratégie a le breakpoint le plus tardif (résiste le plus longtemps)
2. Quelles classes sont les plus vulnérables (classes rares, classes proches visuellement)
3. Quelles tâches déclenchent le plus d'oubli (tâches avec classes visuellement éloignées)
4. Quelle stratégie a le meilleur ratio accuracy / coût mémoire

In [ ]:
print("=" * 75)
print("SYNTHÈSE POUR UNE NOUVELLE MÉTHODE")
print("=" * 75)

# 1. Quelle stratégie a le breakpoint le plus tardif ?
bp_values = {s: results[s]["breakpoint"] for s in STRATEGIES}
best_strat = max(
    [s for s in STRATEGIES if bp_values[s] is not None],
    key=lambda s: bp_values[s],
    default=None,
)
if best_strat is not None:
    print(f"\n✓ Meilleure résistance : {best_strat} (rupture à la tâche {bp_values[best_strat]})")
else:
    print("\n⚠️ Aucune stratégie n'a de rupture — étendre le stream")

# 2. Classes universellement oubliées
all_drops = []
for strat in STRATEGIES:
    mat = results[strat]["acc_matrix"]
    for c in range(mat.shape[1]):
        col = mat[:, c]
        valid = ~np.isnan(col)
        if valid.sum() >= 2:
            drop = np.nanmax(col) - col[-1]
            all_drops.append((c, drop))

from collections import defaultdict
drop_by_class = defaultdict(list)
for c, drop in all_drops:
    drop_by_class[c].append(drop)

mean_drop_per_class = {c: np.mean(ds) for c, ds in drop_by_class.items()}
worst_classes = sorted(mean_drop_per_class.items(), key=lambda x: x[1], reverse=True)[:5]

print("\n📊 Classes les plus vulnérables (toutes stratégies) :")
for c, drop in worst_classes:
    print(f"  {loader.name_label_[c]:15s} : oubli moyen = {drop:.3f}")

# 3. Recommandations pour une nouvelle méthode
print("\n💡 PISTES POUR UNE NOUVELLE MÉTHODE :")
print("  1. Focus sur les classes vulnérables identifiées ci-dessus")
print("  2. Combiner replay (accuracy) + EWC (stabilité) + surprise (sélectivité)")
print("  3. Détecter les tâches visuellement éloignées pour adapter la mémoire")
print("  4. Utiliser la géométrie des embeddings (variété produit) pour")
print("     préserver la structure des classes anciennes")
print("  5. Budget mémoire adaptatif : plus de mémoire pour les classes")
print("     qui s'oublient vite")

print("\n" + "=" * 75)
print("Fichiers produits :")
print(f"  - {out_dir / 'continual_benchmark.csv'}")
print(f"  - {out_dir / 'benchmark_accuracy.png'}")
print(f"  - {out_dir / 'benchmark_forgetting.png'}")
print(f"  - {out_dir / 'benchmark_breakpoint.png'}")
print("=" * 75)

## Récapitulatif du protocole

| Aspect | Valeur |
|---|---|
| Dataset | 28 classes |
| Tâche 0 | 2 classes |
| Tâches suivantes | +2 classes |
| Nombre total de tâches | 14 |
| Stratégies | Replay, EWC, DualMemory |
| Seuil d'oubli catastrophique | 20 % par classe |
| Seuil de rupture | 30 % des classes oubliées |
| Métrique principale | accuracy moyenne sur classes vues |

**Sortie :**
- `continual_benchmark.csv` : tableau récapitulatif
- `benchmark_accuracy.png` : courbes d'accuracy par stratégie
- `benchmark_forgetting.png` : oubli par classe
- `benchmark_breakpoint.png` : fraction de classes oubliées par tâche

**Interprétation :**
- Si aucune rupture → le stream est trop court, ajouter des classes
- Si rupture à la tâche X → c'est le point où la stratégie échoue
- Comparer les breakpoints des 3 stratégies → identifier la plus robuste
- Les classes les plus oubliées → cibles prioritaires pour une nouvelle méthode